# Baseline Model: Logistic Regression
### NYC 311 Service Requests
## Predicting if a ticket is closed within 24 hours

This notebook builds a **logistic regression** model as our baseline: it gives us a reference point to compare against more complex models later.

**Target variable:**
- `Y = 1` if the ticket is closed within 24 hours
- `Y = 0` if the ticket tooks longer (or is still open)



## 1. Imports

In [15]:
import sys
import pandas as pd
import numpy as np


from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


sys.path.insert(0, '/Users/vale/Documents/Machine-Learning-Project/Matteo')
from DS_processing import Process_train_DS, Process_test_DS

print('All imports successful!')

All imports successful!


## 2. Load the raw data



In [16]:

DATA_PATH = '/Users/vale/Desktop/ML project data/'

# Load the training data 
train_raw = pd.read_csv(DATA_PATH + 'train.csv', index_col=0)

# Load the test data ( we use this only at the very end to make final predictions)
test_raw = pd.read_csv(DATA_PATH + 'test.csv', index_col=0)

print(f'Training set: {train_raw.shape[0]} rows, {train_raw.shape[1]} columns')
print(f'Test set:     {test_raw.shape[0]} rows, {test_raw.shape[1]} columns')

Training set: 110930 rows, 40 columns
Test set:     27733 rows, 39 columns


## 3. Process the data

We use Matteo's functions to clean and transform the raw data.
These functions:
- Group complex columns into simpler categories
- Extract date features (hour, day of week, month, weekend)
- Create binary features (Is_Landmark, Is_Taxi)
- Build the target variable Y
- Drop all useless or redundant columns

In [17]:
# Process the training set: this also creates the Y column (our target)
train = Process_train_DS(train_raw)

# Process the test set: no Y column here, just clean features
test = Process_test_DS(test_raw)

print('Training set after processing:')
print(train.shape)
train.head(3)

Dropped 0 rows because their 24-hour window hasn't expired yet.
Training set after processing:
(110930, 14)


,Agency,Incident Zip,Police Precinct,Borough,Open Data Channel Type,Problem_Grouped,Location_Grouped,Is_Landmark,Is_Taxi,Created_Hour,Created_DayOfWeek,Created_Month,Is_Weekend,Y
0,HPD,10011,Precinct 10,MANHATTAN,MOBILE,Housing & Buildings,Residential,0,0,8,1,4,0,1
1,HPD,10025,Precinct 24,MANHATTAN,ONLINE,Housing & Buildings,Residential,0,0,15,3,4,0,0
2,DSNY,11238,Precinct 88,BROOKLYN,PHONE,Sanitation & Garbage,Street/Sidewalk,1,0,12,6,4,1,0


In [18]:
# Quick check: how balanced is our target variable?
# We expect roughly 61% Y=1 and 39% Y=0
print('Target variable distribution:')
print(train['Y'].value_counts(normalize=True).round(3))

Target variable distribution:
Y
1    0.614
0    0.386
Name: proportion, dtype: float64


## 4. Define features and target

I separate the data into:
- **X**: the input features (everything the model can see)
- **y**:the target variable (what the model has to predict)

I also separate categorical columns,from numerical ones,
because they need different preprocessing.

In [19]:
# Separate features (X) from the target (y)
X = train.drop(columns=['Y'])
y = train['Y']

# Categorical columns: need to be converted to numbers
# We use Target Encoding for these 
categorical_cols = [
    'Agency',                  # which city agency handles the request (15 categories)
    'Incident Zip',            # zip code of the incident (222 categories)
    'Police Precinct',         # police precinct (78 categories)
    'Borough',                 # NYC borough (5 boroughs + unspecified)
    'Open Data Channel Type',  # how the request was submitted (4 categories)
    'Problem_Grouped',         # type of problem, grouped into 9 categories
    'Location_Grouped',        # type of location, grouped into 6 categories
]

# Numerical columns: already numbers, just need scaling
numerical_cols = [
    'Created_Hour',      # hour of the day the ticket was created (0-23)
    'Created_DayOfWeek', # day of the week (0=Monday, 6=Sunday)
    'Created_Month',     # month (1-12)
    'Is_Weekend',        # 1 if Saturday or Sunday, 0 otherwise
    'Is_Landmark',       # 1 if the location is a landmark
    'Is_Taxi',           # 1 if the request involves a taxi vehicle
]

print('Categorical features:', categorical_cols)
print('Numerical features:  ', numerical_cols)

Categorical features: ['Agency', 'Incident Zip', 'Police Precinct', 'Borough', 'Open Data Channel Type', 'Problem_Grouped', 'Location_Grouped']
Numerical features:   ['Created_Hour', 'Created_DayOfWeek', 'Created_Month', 'Is_Weekend', 'Is_Landmark', 'Is_Taxi']


## 5. Set up Stratified K-Fold cross-validation

Instead of a single 80/20 split, we use **5-fold cross-validation**:
- The data is split into 5 equal parts (folds)
- The model trains 5 times, each time using a different fold as validation
- We average the 5 accuracy scores for a more reliable estimate

We use `StratifiedKFold` to keep the same Y=1/Y=0 ratio in every fold.

In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-fold cross-validation ready!')

5-fold cross-validation ready!


## 6. Build the preprocessing + model pipeline

Before feeding data into logistic regression, we need to transform all features into numbers on a similar scale.

**Target Encoding** (for categorical columns):
Replaces each category with the average Y value for that category.
Example: if 75% of NYPD tickets are closed in 24h,then encode NYPD as 0.75.
Sklearn handles this safely with cross-fitting to avoid data leakage.

**StandardScaler** (for numerical columns):
Rescales numbers so they all have a mean of 0 and similar range.
Logistic regression is sensitive to feature scale, so this is important.

In [21]:
# ColumnTransformer applies different transformations to different columns simultaneously
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(random_state=42), categorical_cols),
        ('scaler',     StandardScaler(),               numerical_cols),
    ]
)

# Pipeline chains preprocessing and model into one object
# This guarantees the same transformations are applied to train, validation, and test
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,  # more iterations to make sure the model converges
        random_state=42
    ))
])

print('Pipeline ready!')

Pipeline ready!


## 7. Evaluate with K-Fold cross-validation

`cross_val_score` handles everything automatically:
- splits the data into 5 folds
- trains and evaluates the pipeline 5 times
- returns one accuracy score per fold

After this, we fit the final model on **all** the training data to make the best possible test predictions.

In [22]:
all_cols = categorical_cols + numerical_cols

# Fit on ALL training data (not just a subset) for the best possible test predictions
pipeline.fit(X[all_cols], y)

print('Model trained!')

Model trained!


## 8. Evaluate on the validation set

We check how well the model performs on data it has **never seen during training**.
This gives us an honest estimate of real-world performance.

In [23]:
all_cols = categorical_cols + numerical_cols

# cross_val_score trains and evaluates the pipeline 5 times, one per fold
cv_scores = cross_val_score(pipeline, X[all_cols], y, cv=cv, scoring='accuracy')

print(f'CV Accuracy per fold: {cv_scores.round(4)}')
print(f'Mean accuracy:        {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)')
print(f'Std deviation:        {cv_scores.std():.4f}')

CV Accuracy per fold: [0.8365 0.8433 0.8412 0.8376 0.8395]
Mean accuracy:        0.8396 (83.96%)
Std deviation:        0.0025


## 9. Make final predictions on the test set

Now we apply the trained model to the **real test set** (the file with no Y).

In [24]:
# Predict on the test set using the same feature columns
test_predictions = pipeline.predict(test[all_cols])

print(f'Predictions made: {len(test_predictions)} rows')
print(f'Predicted Y=1 (closed in 24h): {test_predictions.sum()} ({test_predictions.mean()*100:.1f}%)')
print(f'Predicted Y=0 (not closed):    {(test_predictions==0).sum()} ({(1-test_predictions.mean())*100:.1f}%)')

Predictions made: 27733 rows
Predicted Y=1 (closed in 24h): 14475 (52.2%)
Predicted Y=0 (not closed):    13258 (47.8%)


## 10. Save the submission file

We format the predictions :
- Index column (row number)
- prediction column (0 or 1)

In [25]:
# Build the submission dataframe 
submission = pd.DataFrame(
    {'prediction': test_predictions},
    index=test.index
)

# Save to CSV (memo:  upload to Blackboard)
submission.to_csv(DATA_PATH + 'submission_valeria.csv')

print('Submission saved to:', DATA_PATH + 'submission_valeria.csv')
print('\nFirst few rows:')
submission.head()

Submission saved to: /Users/vale/Desktop/ML project data/submission_valeria.csv

First few rows:


,prediction
0,0
1,0
2,0
3,0
4,0
